# SENTINEL-X: Getting Started Tutorial

This notebook walks through the core SENTINEL-X pipeline:

1. **Install** dependencies (skip if already installed)
2. **Simulate** a spacecraft fault scenario
3. **Train** a DQN agent from scratch
4. **Evaluate** the trained policy and visualise recovery actions
5. **Export** the policy to TFLite int8 for embedded deployment
6. **Federated learning** with a 3-spacecraft swarm

**Expected runtime:** ≈ 3–5 minutes (CPU), ≈ 1–2 minutes (GPU).

---
**Repository:** https://github.com/danielnovais-tech/SENTINEL-X  
**License:** Apache 2.0

## Step 0 – Install dependencies

Skip this cell if you are running inside the cloned repository with the
virtual environment already set up.

In [ ]:
# Uncomment and run once
# !pip install numpy tensorflow matplotlib scikit-learn

## Step 1 – Imports

In [ ]:
import os
import sys
import random
import tempfile

import numpy as np
import matplotlib.pyplot as plt

# Suppress TF log noise
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# Add the repo root to the Python path if running from the notebooks/ folder
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), '..'))

import sentinel_x_advanced as sx

print("SENTINEL-X imported successfully!")
print(f"Available scenarios: {[s for s in dir(sx) if 'scenario' in s.lower() or 'mission' in s.lower()]}")

## Step 2 – Simulate fault scenarios

SENTINEL-X models six fault generators:
- **MemoryArray** – bit-flip errors (SEU / MBU)
- **Sensor** – stuck values, noise spikes
- **ThermalSubsystem** – over-temperature events
- **PowerSubsystem** – low-voltage / brownout
- **AttitudeControlSubsystem** – rate gyro drift
- **CommSubsystem** – link quality degradation

In [ ]:
# Create one instance of each fault subsystem
mem    = sx.MemoryArray(size=1024)
sensor = sx.SensorSubsystem()
therm  = sx.ThermalSubsystem()
power  = sx.PowerSubsystem()
adcs   = sx.AttitudeControlSubsystem()
comm   = sx.CommSubsystem()

# Inject a few faults and print the resulting state
rng = np.random.default_rng(42)

for step in range(5):
    mem.step(rng)
    sensor.step(rng)
    therm.step(rng)
    power.step(rng)
    adcs.step(rng)
    comm.step(rng)

state = sx.get_state_vector(mem, sensor, therm, power, adcs, comm)
print("State vector (11 features):")
feature_names = [
    "mem_errors", "parity_err", "sensor_dev", "health_flag",
    "thermal_fault", "time_since_recovery", "power_level",
    "attitude_rate", "comm_quality", "sensor_stuck", "power_critical"
]
for name, val in zip(feature_names, state):
    print(f"  {name:25s}: {val:.4f}")

## Step 3 – Create the environment and a DQN agent

In [ ]:
# Spacecraft environment (OpenAI Gym-compatible)
env = sx.SpacecraftEnv()
print(f"Observation space: {env.observation_space}")
print(f"Action space     : {env.action_space}")
print(f"Actions          : DO_NOTHING=0  RESTART=1  SWITCH_REDUNDANT=2  SAFE_MODE=3")

# DQN agent with experience replay
agent = sx.DQNAgent(
    state_dim=env.observation_space.shape[0],
    action_dim=env.action_space.n,
)
print(f"\nAgent model summary:")
agent.model.summary()

## Step 4 – Train the agent

We run 20 training episodes here for speed.  For a production policy use 100–500 episodes.

In [ ]:
N_TRAIN_EPISODES = 20
MAX_STEPS        = 200

episode_rewards = []
epsilon_history = []

for ep in range(N_TRAIN_EPISODES):
    state, _ = env.reset()
    total_reward = 0.0

    for step in range(MAX_STEPS):
        action = agent.act(state)
        next_state, reward, done, truncated, _ = env.step(action)
        agent.remember(state, action, reward, next_state, done)
        agent.replay()
        state = next_state
        total_reward += reward
        if done or truncated:
            break

    episode_rewards.append(total_reward)
    epsilon_history.append(agent.epsilon)

    if (ep + 1) % 5 == 0:
        print(f"Episode {ep+1:3d}  reward={total_reward:8.2f}  epsilon={agent.epsilon:.3f}")

print(f"\nFinal mean reward (last 5 eps): {np.mean(episode_rewards[-5:]):.2f}")

## Step 5 – Visualise the reward curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(episode_rewards, linewidth=2, color='steelblue')
# Smooth with 5-ep rolling average
if len(episode_rewards) >= 5:
    smooth = np.convolve(episode_rewards, np.ones(5)/5, mode='valid')
    axes[0].plot(range(4, len(episode_rewards)), smooth,
                 linewidth=2, color='crimson', label='5-ep average')
    axes[0].legend()
axes[0].set_title('Episode Reward')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Total Reward')
axes[0].grid(alpha=0.3)

axes[1].plot(epsilon_history, linewidth=2, color='darkorange')
axes[1].set_title('Exploration Rate (ε)')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Epsilon')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/sentinel_x_training.png', dpi=120)
plt.show()
print("Plot saved to /tmp/sentinel_x_training.png")

## Step 6 – Evaluate the trained policy and visualise actions

In [ ]:
ACTION_NAMES = ["DO_NOTHING", "RESTART", "SWITCH_REDUNDANT", "SAFE_MODE"]
safety = sx.SafetyMonitor()

state, _ = env.reset()
action_log = []
reward_log = []

for step in range(50):
    action = agent.act(state, epsilon=0.0)       # greedy
    action = safety.veto(state, action)           # safety override
    next_state, reward, done, truncated, _ = env.step(action)
    action_log.append(action)
    reward_log.append(reward)
    state = next_state
    if done or truncated:
        break

# Count action distribution
from collections import Counter
dist = Counter(action_log)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar([ACTION_NAMES[a] for a in sorted(dist)],
            [dist[a] for a in sorted(dist)],
            color=['steelblue', 'orange', 'green', 'crimson'])
axes[0].set_title('Action Distribution (eval episode)')
axes[0].set_xlabel('Action')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=15)
axes[0].grid(axis='y', alpha=0.3)

axes[1].plot(reward_log, linewidth=2, color='steelblue')
axes[1].axhline(0, color='gray', linestyle='--', linewidth=1)
axes[1].set_title('Per-Step Reward (eval episode)')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Reward')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/sentinel_x_eval.png', dpi=120)
plt.show()

print(f"Eval episode total reward: {sum(reward_log):.2f}")

## Step 7 – Safety verification with SafetyMonitor

In [ ]:
# Construct a deliberately unsafe state: critical power + health flag
unsafe_state = np.zeros(11)
unsafe_state[3] = 1.0   # health_flag = True
unsafe_state[6] = 0.05  # power_level very low

# Agent would choose DO_NOTHING (index 0) with an untrained policy
raw_action  = 0  # DO_NOTHING
safe_action = safety.veto(unsafe_state, raw_action)

print(f"Raw agent action : {ACTION_NAMES[raw_action]}")
print(f"After SafetyMonitor veto: {ACTION_NAMES[safe_action]}")
print("\nSafetyMonitor prevents DO_NOTHING when health_flag is set.")

## Step 8 – Export to TFLite int8 for embedded deployment

In [ ]:
import tempfile, os

with tempfile.TemporaryDirectory() as tmpdir:
    model_path = os.path.join(tmpdir, "sentinel_x_model_int8.tflite")
    sx.export_tflite_int8(agent, model_path)

    size_kb = os.path.getsize(model_path) / 1024
    print(f"TFLite int8 model exported: {model_path}")
    print(f"Model size: {size_kb:.1f} KB  (target: < 256 KB for STM32H7)")

    # Load back and run one inference
    import tensorflow as tf
    interp = tf.lite.Interpreter(model_path=model_path)
    interp.allocate_tensors()
    inp = interp.get_input_details()[0]
    out = interp.get_output_details()[0]

    test_state = state[np.newaxis].astype(np.float32)
    interp.set_tensor(inp['index'], test_state)
    interp.invoke()
    q_values = interp.get_tensor(out['index'])[0]

    print(f"\nTFLite Q-values: {q_values}")
    print(f"Recommended action: {ACTION_NAMES[int(np.argmax(q_values))]}")

## Step 9 – Federated learning with a 3-spacecraft swarm

Each spacecraft trains independently, then agents average their weights via
Federated Averaging (FedAvg).  The aggregated policy should outperform any
single-spacecraft policy.

In [ ]:
N_SPACECRAFT     = 3
FED_ROUNDS       = 3
EPISODES_PER_ROUND = 5

swarm = sx.FederatedSwarm(
    n_agents=N_SPACECRAFT,
    state_dim=env.observation_space.shape[0],
    action_dim=env.action_space.n,
)

fed_rewards = []

for fed_round in range(FED_ROUNDS):
    round_rewards = swarm.train_round(
        env=env,
        episodes_per_agent=EPISODES_PER_ROUND,
        max_steps=200,
    )
    mean_r = np.mean(round_rewards)
    fed_rewards.extend(round_rewards)
    print(f"Fed round {fed_round+1}/{FED_ROUNDS}  mean reward={mean_r:.2f}")

print(f"\nFederated mean reward (all rounds): {np.mean(fed_rewards):.2f}")
print(f"Single-agent baseline: {np.mean(episode_rewards):.2f}")

## Summary

You have successfully:

| Step | Result |
|------|--------|
| Simulated fault scenario | ✅ 11-dimensional state vector |
| Trained DQN agent | ✅ {N} episodes |
| Applied SafetyMonitor | ✅ DO_NOTHING vetoed on unsafe states |
| Exported to TFLite int8 | ✅ < 256 KB, runs on STM32H7 |
| Federated learning (3 s/c) | ✅ FedAvg weight sharing |

### Next steps

- **Hardware deployment:** see [`docs/hardware_integration.md`](../docs/hardware_integration.md)
- **Custom fault models:** see [`docs/custom_mission_tutorial.md`](../docs/custom_mission_tutorial.md)
- **Formal verification:** see [`docs/formal_verification_external.md`](../docs/formal_verification_external.md)
- **Long-term autonomy:** see [`docs/continuous_learning.md`](../docs/continuous_learning.md)
- **Benchmarking baselines:** see [`docs/benchmarking.md`](../docs/benchmarking.md)
- **Full roadmap:** [`ROADMAP.md`](../ROADMAP.md)